In [1]:
!wget https://jdbc.postgresql.org/download/postgresql-42.7.0.jar

--2026-05-01 20:11:40--  https://jdbc.postgresql.org/download/postgresql-42.7.0.jar
Resolving jdbc.postgresql.org (jdbc.postgresql.org)... 72.32.157.228, 2001:4800:3e1:1::228
Connecting to jdbc.postgresql.org (jdbc.postgresql.org)|72.32.157.228|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1077325 (1.0M) [application/java-archive]
Saving to: ‘postgresql-42.7.0.jar.6’

postgresql-42.7.0.j 100%[===================>]   1.03M   950KB/s    in 1.1s    

2026-05-01 20:11:43 (950 KB/s) - ‘postgresql-42.7.0.jar.6’ saved [1077325/1077325]



In [2]:
import os
from pyspark.sql import SparkSession, functions as F


jar_path = f"{os.getcwd()}/postgresql-42.7.0.jar"

spark = (SparkSession.builder.appName("Instacart-Retail")
         .config("spark.jars", jar_path) 
         .config("spark.driver.extraClassPath", jar_path)   # <--- Force Driver
         .config("spark.executor.extraClassPath", jar_path) # <--- Force Driver
         .config("spark.driver.memory", "4g")
         .getOrCreate())

BRONZE = "/home/jovyan/work/data/bronze"
order_products_prior = spark.read.parquet(f"{BRONZE}/order_products_prior")
products = spark.read.parquet(f"{BRONZE}/products")
departments = spark.read.parquet(f"{BRONZE}/departments")
orders = spark.read.parquet(f"{BRONZE}/orders")


In [3]:
# ============================================
# Q1: Top 10 most ordered products
# (Business Q: SKU ไหนขายดีสุด)
# ============================================
top_products = (order_products_prior
    .groupBy("product_id")
    .agg(F.count("*").alias("order_count"))
    .join(products, on="product_id")
    .orderBy(F.desc("order_count"))
    .limit(10))

top_products.show(truncate=False)
# คาดว่าจะเห็น: Banana, Bag of Organic Bananas, Organic Strawberries...

+----------+-----------+----------------------+--------+-------------+
|product_id|order_count|product_name          |aisle_id|department_id|
+----------+-----------+----------------------+--------+-------------+
|24852     |472565     |Banana                |24      |4            |
|13176     |379450     |Bag of Organic Bananas|24      |4            |
|21137     |264683     |Organic Strawberries  |24      |4            |
|21903     |241921     |Organic Baby Spinach  |123     |4            |
|47209     |213584     |Organic Hass Avocado  |24      |4            |
|47766     |176815     |Organic Avocado       |24      |4            |
|47626     |152657     |Large Lemon           |24      |4            |
|16797     |142951     |Strawberries          |24      |4            |
|26209     |140627     |Limes                 |24      |4            |
|27845     |137905     |Organic Whole Milk    |84      |16           |
+----------+-----------+----------------------+--------+-------------+



In [4]:
# ============================================
# Q2: Reorder rate per department
# (Business Q: department ไหนลูกค้าซื้อซ้ำบ่อยสุด — สำคัญใน FMCG)
# ============================================
reorder_by_dept = (order_products_prior
    .join(products, on="product_id")
    .join(departments, on="department_id")
    .groupBy("department")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("reordered").alias("reorder_count"),
        F.round(F.avg("reordered"), 3).alias("reorder_rate")
    )
    .orderBy(F.desc("reorder_rate")))

reorder_by_dept.show(25, truncate=False)
# Insight: dairy eggs, beverages, produce → reorder rate สูง (consumable)
#          personal care, household → ต่ำ (durable)

+---------------+------------+-------------+------------+
|department     |total_orders|reorder_count|reorder_rate|
+---------------+------------+-------------+------------+
|dairy eggs     |5414016     |3627221      |0.67        |
|beverages      |2690129     |1757892      |0.653       |
|produce        |9479291     |6160710      |0.65        |
|bakery         |1176787     |739188       |0.628       |
|deli           |1051249     |638864       |0.608       |
|pets           |97724       |58760        |0.601       |
|babies         |423802      |245369       |0.579       |
|bulk           |34573       |19950        |0.577       |
|snacks         |2887550     |1657973      |0.574       |
|alcohol        |153696      |87595        |0.57        |
|meat seafood   |708931      |402442       |0.568       |
|breakfast      |709569      |398013       |0.561       |
|frozen         |2236432     |1211890      |0.542       |
|dry goods pasta|866627      |399581       |0.461       |
|canned goods 

In [5]:
# ============================================
# Q3: Order pattern by day of week + hour
# (Business Q: peak hour ไหน — สำหรับ workforce planning)
# ============================================
dow_map = {0:"Sun", 1:"Mon", 2:"Tue", 3:"Wed", 4:"Thu", 5:"Fri", 6:"Sat"}
dow_expr = F.create_map([F.lit(x) for kv in dow_map.items() for x in kv])

orders_pattern = (orders
    .withColumn("dow_name", dow_expr[F.col("order_dow")])
    .groupBy("dow_name", "order_hour_of_day")
    .agg(F.count("*").alias("order_count"))
    .orderBy("dow_name", "order_hour_of_day"))

orders_pattern.show(20)

+--------+-----------------+-----------+
|dow_name|order_hour_of_day|order_count|
+--------+-----------------+-----------+
|     Fri|                0|       3189|
|     Fri|                1|       1672|
|     Fri|                2|       1016|
|     Fri|                3|        841|
|     Fri|                4|        910|
|     Fri|                5|       1574|
|     Fri|                6|       4866|
|     Fri|                7|      13434|
|     Fri|                8|      24015|
|     Fri|                9|      34232|
|     Fri|               10|      38313|
|     Fri|               11|      37915|
|     Fri|               12|      35714|
|     Fri|               13|      36296|
|     Fri|               14|      37407|
|     Fri|               15|      37508|
|     Fri|               16|      35860|
|     Fri|               17|      29955|
|     Fri|               18|      24310|
|     Fri|               19|      18741|
+--------+-----------------+-----------+
only showing top

In [6]:
# ============================================
# Q4: Window function — ลำดับสินค้าใน basket
# (Business Q: สินค้าที่ลูกค้าหยิบเป็นชิ้นแรก = "destination product")
# ============================================
from pyspark.sql.window import Window

# สินค้าชิ้นแรกของทุก order
first_items = (order_products_prior
    .filter(F.col("add_to_cart_order") == 1)
    .join(products, on="product_id")
    .groupBy("product_name")
    .agg(F.count("*").alias("first_pick_count"))
    .orderBy(F.desc("first_pick_count"))
    .limit(15))

first_items.show(truncate=False)
# Insight: Banana ถูกหยิบเป็นชิ้นแรกบ่อย → "anchor product"

+--------------------------+----------------+
|product_name              |first_pick_count|
+--------------------------+----------------+
|Banana                    |110916          |
|Bag of Organic Bananas    |78988           |
|Organic Whole Milk        |30927           |
|Organic Strawberries      |27975           |
|Organic Hass Avocado      |24116           |
|Organic Baby Spinach      |23543           |
|Organic Avocado           |22398           |
|Spring Water              |16822           |
|Strawberries              |16366           |
|Organic Raspberries       |14393           |
|Sparkling Water Grapefruit|13733           |
|Organic Half & Half       |12676           |
|Large Lemon               |12316           |
|Soda                      |11770           |
|Organic Reduced Fat Milk  |9885            |
+--------------------------+----------------+



In [7]:
# ============================================
# Compare: regular join vs broadcast join
# ============================================

# Regular join — Spark จะ shuffle ทั้ง 2 sides
big_join = order_products_prior.join(products, on="product_id")
big_join.explain(mode="formatted")
# จะเห็น "SortMergeJoin" → มี shuffle exchange

# Broadcast join — products เล็ก (~2MB) → broadcast ไปทุก executor
from pyspark.sql.functions import broadcast
fast_join = order_products_prior.join(broadcast(products), on="product_id")
fast_join.explain(mode="formatted")
# จะเห็น "BroadcastHashJoin" → ไม่มี shuffle ฝั่ง products

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- BroadcastHashJoin Inner BuildRight (6)
      :- Filter (2)
      :  +- Scan parquet  (1)
      +- BroadcastExchange (5)
         +- Filter (4)
            +- Scan parquet  (3)


(1) Scan parquet 
Output [4]: [order_id#0, product_id#1, add_to_cart_order#2, reordered#3]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/work/data/bronze/order_products_prior]
PushedFilters: [IsNotNull(product_id)]
ReadSchema: struct<order_id:int,product_id:int,add_to_cart_order:int,reordered:int>

(2) Filter
Input [4]: [order_id#0, product_id#1, add_to_cart_order#2, reordered#3]
Condition : isnotnull(product_id#1)

(3) Scan parquet 
Output [4]: [product_id#8, product_name#9, aisle_id#10, department_id#11]
Batched: true
Location: InMemoryFileIndex [file:/home/jovyan/work/data/bronze/products]
PushedFilters: [IsNotNull(product_id)]
ReadSchema: struct<product_id:int,product_name:string,aisle_id:string,department_id:string>

(4) Filter
I

In [8]:
# ============================================
# Performance test: เปรียบเทียบ time
# ============================================
import time

# Test 1: regular
start = time.time()
order_products_prior.join(products, "product_id").count()
print(f"Regular join: {time.time()-start:.2f}s")

# Test 2: broadcast (clear cache ก่อน)
spark.catalog.clearCache()
start = time.time()
order_products_prior.join(broadcast(products), "product_id").count()
print(f"Broadcast join: {time.time()-start:.2f}s")

Regular join: 0.32s
Broadcast join: 0.21s


In [9]:
spark = (SparkSession.builder
    .appName("write-postgres")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.0")
    .config("spark.driver.memory", "4g")
    .getOrCreate())

# Reload data (ถ้า restart kernel)
# ... [reload code] ...

# Write department-level KPI ลง warehouse
reorder_by_dept.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://pg-warehouse:5432/warehouse") \
    .option("dbtable", "kpi_reorder_by_department") \
    .option("user", "dataeng") \
    .option("password", "dataeng") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

print("✅ Written to Postgres")

✅ Written to Postgres
